In [ ]:
# [CELL 1] Setup: Generate sample PHP scripts with OWASP Top 10 vulnerabilities & safe patterns
import os

print("[*] Creating sample vulnerable PHP file for security analysis...")
vulnerable_code = '''<?php
// 1. SQL Injection
$id = $_GET['id'];
$pdo = new PDO('sqlite::memory:');
$pdo->query("SELECT * FROM users WHERE id = " . $id);

// 2. Cross-Site Scripting (XSS)
$name = $_POST['name'];
echo "<h1>Welcome, " . $name . "</h1>";

// 3. Remote Code Execution (RCE)
$cmd = $_GET['cmd'];
system("ping -c 2 " . $cmd);

// 4. Insecure Deserialization
$token = $_COOKIE['auth_token'];
$user = unserialize($token);
'''

with open("sample_vulnerable.php", "w") as f:
    f.write(vulnerable_code)

print("[✓] Sample vulnerable file written to sample_vulnerable.php")
!php -v

In [ ]:
# [CELL 2] Execute static scan with CLI & fail-on security gate
!php bin/eidcloud-sec scan sample_vulnerable.php --fail-on=high || true

In [ ]:
# [CELL 3] Export JSON report and parse findings with Python
import json
import subprocess

result = subprocess.run(
    ["php", "bin/eidcloud-sec", "scan", "sample_vulnerable.php", "--json"],
    capture_output=True,
    text=True
)

report = json.loads(result.stdout)
summary = report["summary"]

print("=" * 70)
print(f"EIDCLOUD SECURITY SCAN REPORT - {summary['total_findings']} VULNERABILITIES DETECTED")
print("=" * 70)
print(f"Files Scanned: {summary['files_scanned']} | LOC: {summary['lines_scanned']} | Time: {summary['duration_seconds']}s")
print(f"Severity Breakdown: {summary['counts_by_severity']}\n")

for idx, f in enumerate(report["findings"], start=1):
    print(f"[{f['severity']}] #{idx} {f['rule_id']}: {f['title']}")
    print(f"  Location:    {f['file']}:{f['line']}")
    print(f"  Remediation: {f['remediation']}")
    if f.get("diff_suggestion"):
        print("  Suggested Patch:")
        for line in f["diff_suggestion"].splitlines():
            print(f"    {line}")
    print("-" * 70)